# Interpret a vision model

**What it does.** Relate a classifier's scores back to the measured features.

**When to use it.** To put a number on what a network learned, in terms of features a reviewer can read.

**What you get.** Feature-versus-score correlations and their ranking.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.submodules.interpret_vision_model`

```
interpret_vision_model(settings=None)
```

Explain a spacr vision-model score by ranking which morphology / intensity features drive it.

In [ ]:
from spacr.submodules import interpret_vision_model

## 3. Settings and API reference

Read the descriptions here, then edit only the values in the next cell. Defaults and descriptions are generated from the installed spaCR version, so the notebook stays aligned with the API.

### [`spacr.submodules.interpret_vision_model`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.interpret_vision_model)

> This function does not yet expose a settings factory. The list below is recovered from direct settings access in its source, so a key used only on a dynamic branch may be absent.

- **`channels`** — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3].
- **`feature_importance`** — (bool) - Fit a random forest against the score column and plot its impurity-based importances. Fast and always available, but biased toward high-cardinality and correlated features, so read the result as a shortlist rather than a ranking. Turning it off skips that plot and its two grouped-by-compartment and grouped-by-channel companions. Default True.
- **`include_all`** — (bool) - When grouping feature importances by compartment and by channel, also emit an 'all' row totalling the features that belong to no single compartment or channel. With it off those features are simply absent from the grouped plots, so the bars no longer sum to the whole and a large shared contribution is invisible. Default False.
- **`n_jobs`** — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`nuclei_limit`** — (int, bool, or None) - Cap on nuclei per cell, applied when the per-object tables are merged. None disables the filter, True keeps only single-nucleus cells, and an integer N keeps cells with N or fewer. Cells over the cap are dropped from the merged table entirely. Do NOT pass False: it is read as 0 and removes every cell, leaving an empty analysis rather than an error. Default None.
- **`pathogen_limit`** — (int, bool, or None) - Maximum pathogens per cell. True or 1 = single pathogen only; None or False = no limit; int = custom limit. Default varies by module (1, 3, 10 or 1000 depending on the factory that fills it), so check the module's own settings rather than assuming one value.
- **`permutation_importance`** — (bool) - Re-score the fitted forest with each feature shuffled in turn, ten repeats, and rank by how far the score falls. Much slower than feature_importance and far more trustworthy, because a feature that shuffles harmlessly was not being used. Turn it on when a shortlist has to become a claim. Default False.
- **`save`** — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.
- **`score_column`** — (str) - Which column of the prediction CSV holds the CNN score that Explain CV and the hit-investigation montages read. The regression module no longer has this setting: it fits dependent_variable and simulates the minimum cell count on that same column, so one measurement cannot be named two ways there. Default 'cv_predictions'.
- **`scores`** — (str, path) - CSV of per-object model scores to interpret, joined to the measurements on plateID, rowID, columnID, fieldID and object_label. This is a classification run's output, and the interpretation explains THESE scores - a CSV from a different model or plate yields a confident explanation of the wrong thing rather than an error. Default None.
- **`shap`** — (bool) - Compute SHAP values, which attribute each individual prediction to each feature instead of ranking features overall. It is the only one of the three that can explain a single object, and by far the slowest - see shap_sample before enabling it on a full plate. Default False.
- **`shap_sample`** — (bool) - Run SHAP on a subsample rather than every object. SHAP cost grows with the row count, so on a full plate this is the difference between minutes and hours, and the feature ranking is stable long before the individual values are. Turn it off only when a specific object's attribution has to be exact. Default True.
- **`src`** — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`tables`** — (list) - Measurement tables read from each plate's database and merged into one analysis frame. Only 'cell', 'nucleus', 'pathogen', 'cytoplasm' and 'png_list' are actually merged. Any other name -- INCLUDING 'organelle', which the measure step does write -- is loaded and then dropped, so asking for it costs time and returns nothing, with no warning that the table you wanted is missing from the result. Default ['cell', 'nucleus', 'pathogen', 'cytoplasm'].
- **`top_features`** — (int) - Feature cap in the ML screen analysis: how many rows the feature-importance and permutation-importance bar plots show, and how many top-ranked features the SHAP refit and its summary plot use. It is also the k of the SelectKBest pruning applied before the model is fitted, but only when prune_features is True - with prune_features at its default False the classifier trains on every feature and this is reporting/SHAP scope only. Raise for a fuller picture, lower for readable plots. Default 30.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    'channels': None,
    'feature_importance': None,
    'include_all': None,
    'n_jobs': None,
    'nuclei_limit': None,
    'pathogen_limit': None,
    'permutation_importance': None,
    'save': None,
    'score_column': None,
    'scores': None,
    'shap': None,
    'shap_sample': None,
    'src': None,
    'tables': None,
    'top_features': None,
}

In [ ]:
interpret_vision_model(settings)

## Where the output went

Feature-versus-score correlations and their ranking.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.